In [1]:
pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader


In [3]:
def load_all_pdfs():
  folder_path = "data/pdfs"
  num_docs = 0
  all_docs = []

  for filename in os.listdir(folder_path):
    if filename.lower().endswith('.pdf'):
      pdf_path = os.path.join(folder_path, filename)
      loader = PyPDFLoader(pdf_path)
      doc = loader.load()
      all_docs.extend(doc)
      num_docs += 1

  print("total pdfs:", num_docs)
  print("total pages:",len(all_docs))
  return all_docs
all_pdf_documents = load_all_pdfs()
print(all_pdf_documents)

total pdfs: 1
total pages: 102
[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Acrobat PDFMaker 18 for Word', 'creationdate': '2017-11-23T17:47:10+05:30', 'company': '', 'created': 'D:20171115', 'lastsaved': 'D:20171115', 'moddate': '2017-11-23T17:58:32+05:30', 'sourcemodified': 'D:20171123121634', 'source': 'data/pdfs/100 aptitude tricks (1).pdf', 'total_pages': 102, 'page': 0, 'page_label': '1'}, page_content='IBPS SBI UPSC RBI RRB SSC \nBSNL NTPC BHEL TNEB NLC CBI \nTANCET GATE CAT MAT XAT SAT \nPOLICE ARMY GMAT GRE POST IPM \nAll Central and State Government Exams \nCampus Recruitment Tests'), Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Acrobat PDFMaker 18 for Word', 'creationdate': '2017-11-23T17:47:10+05:30', 'company': '', 'created': 'D:20171115', 'lastsaved': 'D:20171115', 'moddate': '2017-11-23T17:58:32+05:30', 'sourcemodified': 'D:20171123121634', 'source': 'data/pdfs/100 aptitude tricks (1).pdf', 'total_pages': 102, 'page': 1, '

In [4]:
print(type(all_pdf_documents))

<class 'list'>


In [5]:
!pip install langchain_text_splitters

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents , chunk_size=500 , chunk_overlap=50):
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=chunk_size,
      chunk_overlap=chunk_overlap
  )
  chunk_docs = text_splitter.split_documents(documents)
  print(chunk_docs)
  return chunk_docs

chunks = split_docs(all_pdf_documents)
print(len(chunks))
chunks[50]

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Acrobat PDFMaker 18 for Word', 'creationdate': '2017-11-23T17:47:10+05:30', 'company': '', 'created': 'D:20171115', 'lastsaved': 'D:20171115', 'moddate': '2017-11-23T17:58:32+05:30', 'sourcemodified': 'D:20171123121634', 'source': 'data/pdfs/100 aptitude tricks (1).pdf', 'total_pages': 102, 'page': 0, 'page_label': '1'}, page_content='IBPS SBI UPSC RBI RRB SSC \nBSNL NTPC BHEL TNEB NLC CBI \nTANCET GATE CAT MAT XAT SAT \nPOLICE ARMY GMAT GRE POST IPM \nAll Central and State Government Exams \nCampus Recruitment Tests'), Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Acrobat PDFMaker 18 for Word', 'creationdate': '2017-11-23T17:47:10+05:30', 'company': '', 'created': 'D:20171115', 'lastsaved': 'D:20171115', 'moddate': '2017-11-23T17:58:32+05:30', 'sourcemodified': 'D:20171123121634', 'source': 'data/pdfs/100 aptitude tricks (1).pdf', 'total_pages': 102, 'page': 1, 'page_label': '2'}, page_content

Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Acrobat PDFMaker 18 for Word', 'creationdate': '2017-11-23T17:47:10+05:30', 'company': '', 'created': 'D:20171115', 'lastsaved': 'D:20171115', 'moddate': '2017-11-23T17:58:32+05:30', 'sourcemodified': 'D:20171123121634', 'source': 'data/pdfs/100 aptitude tricks (1).pdf', 'total_pages': 102, 'page': 28, 'page_label': '29'}, page_content='S.P = 45000[0.8]2 \nS.P = 28800 \nSelling price after two years \n \n= Rs. 28800 \nwww.iascgl.com')

In [7]:
from sentence_transformers import SentenceTransformer


In [8]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):

        self.model_name = model_name
        print("loading model...", self.model_name)
        self.model = SentenceTransformer(model_name)
        print("embedding dimensions=", self.model.get_embedding_dimension())

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embedding shape:", embeddings.shape)
        return embeddings



In [9]:
embedding_manager = EmbeddingManager()



loading model... all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding dimensions= 384


In [10]:
import chromadb
import uuid

In [11]:
class VectorStoreManager:
  def __init__(self, persist_directory= "data/vector_store", collection_name="pdf_documents"):
    self.collection_name=collection_name
    self.persist_directory=persist_directory
    self.collection=None
    self.client = None

    self._initialize_store()

  def _initialize_store(self):
    os.makedirs(self.persist_directory, exist_ok=True)
#creating client
    self.client = chromadb.PersistentClient(path=self.persist_directory)
#creating Collection

    self.collection= self.client.get_or_create_collection(
       name=self.collection_name,
       metadata={"description":"Vector Store Collection for pdf embedding in RAG"}
    )
    print("Initilized the vector store collection", self.collection_name)
    print("docs in Collcetion ", self.collection.count())

  def add_documents(self, documents , embeddings):
    if len(documents) != len(embeddings):
      raise ValueError("Number of documents does not match with the num of Embeddings")

    ids=[]
    all_metadata=[]
    document_content=[]
    embeddings_list = []

    for i , (doc , embedding) in enumerate(zip(documents,embeddings)):
      doc_id = f"doc_{uuid.uuid4()}"
      ids.append(doc_id)

      metadata=dict(doc.metadata)
      metadata["doc_index"]=i
      metadata["content_length"] = len(doc.page_content)
      all_metadata.append(metadata)

      document_content.append(doc.page_content)
      embeddings_list.append(embedding.tolist())

      self.collection.add(
          ids=ids,
          documents=document_content,
          metadatas=all_metadata,
          embeddings=embeddings_list
      )
    print("Total document add in vector store:",len(document_content))
    print("Docs in Collection :", self.collection.count())


In [12]:
vector_store = VectorStoreManager()

Initilized the vector store collection pdf_documents
docs in Collcetion  692


In [13]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embeddings)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

embedding shape: (173, 384)
Total document add in vector store: 173
Docs in Collection : 865


In [14]:
from sklearn.metrics.pairwise import cosine_similarity



In [15]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs = []

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no document found")

        return retrieved_docs

In [26]:
rag_retriever = RAGRetriever(
    embedding_manager,
    vector_store
)

print("Retriever created successfully ✅")

Retriever created successfully ✅


In [17]:
reg_retriever.retrieve("how to sholve Ratio Questions")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_ea04eecc-f89a-4349-bc9c-199c2add8a4f',
  'document': 'Shortcut # 8 – Ratio and Proportion \nAdding and removing quantities. \n \nQuestion: \n \nContainer 1 has milk and water in the ratio 2 : 5. After adding 4 liters of pure \nwater from container 2, the ratio between milk and water in container 1 \nbecame 1 : 3. Find the quantity of milk in the container. \n \nAnswer: \nLet us assume the actual quantity of milk and water as 2x and 5x. \nNew quantity of water = 5x + 4 \n2x/(5x + 4) = 1/3 \n6x = 5x + 4 \nx = 4 \nQuantity of milk = 2x \n= 2(4) \n= 8 liters.',
  'metadata': {'producer': 'Adobe PDF Library 15.0',
   'doc_index': 13,
   'moddate': '2017-11-23T17:58:32+05:30',
   'created': 'D:20171115',
   'company': '',
   'sourcemodified': 'D:20171123121634',
   'creator': 'Acrobat PDFMaker 18 for Word',
   'page_label': '10',
   'creationdate': '2017-11-23T17:47:10+05:30',
   'source': 'data/pdfs/100 aptitude tricks (1).pdf',
   'total_pages': 102,
   'content_length': 490,


In [18]:
!pip install langchain-groq

In [19]:
API_GROQ_KEY= "your api"


In [20]:
from langchain_groq import ChatGroq

In [21]:
# llm = ChatGroq(
#     grok_api_key = API_GROQ_KEY,
#     model = "qwen/qwen3-32b",
#     temperature=0.1,
#     max_tokens=1024
# )

llm = ChatGroq(
    groq_api_key=API_GROQ_KEY,
    model="qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=1024
)

In [30]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # prompt => context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context},
                Query: {query}
            """

    response = llm.invoke([prompt]) # expecting a list as prompt
    return response.content

In [35]:
answer = generate_output(
    "how to solve ratio questions?",
    rag_retriever,
    llm
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape: (1, 384)
retrieved 3 documents


In [36]:
print(answer)

<think>
Okay, the user is asking how to solve ratio questions. Let me look at the context provided. The example given is about a container with milk and water in a ratio of 2:5, then adding 4 liters of water to change the ratio to 1:3. The solution uses variables (2x and 5x) and sets up an equation after adding the water.

First, I need to explain the general approach to ratio problems. The key steps here are understanding the initial ratio, accounting for any changes (like adding or removing quantities), and setting up equations based on the new ratio. The example uses algebra to solve for x, which gives the quantities of milk and water.

I should outline the steps methodically. Start by assigning variables based on the original ratio. Then, adjust the quantities according to the problem's changes. Set up a proportion equation using the new ratio and solve for the variable. Finally, use the variable to find the required quantity.

Wait, the user might not just want the steps for this 